<a href="https://colab.research.google.com/github/Viggo-Kristensen/kaggle-competitions/blob/main/student_test_score_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Problem Definition**

Predict students `exam_score`'s based on features like `class_attendance`, `study_hours` and `sleep_quality`

## **Imports**

In [ ]:
import pandas as pd
import os
from sklearn.base import TransformerMixin, BaseEstimator
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

## **Dataset**

### Uploading Files

In [ ]:
!mkdir -p /root/.kaggle
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [ ]:
# upload kaggle.json ONCE
from google.colab import files
files.upload()

!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle competitions download -c playground-series-s6e1

Saving kaggle.json to kaggle.json
100% 13.8M/13.8M [00:00<00:00, 57.4MB/s]



### Unzipping Files

In [ ]:
!unzip -q playground-series-s6e1.zip -d data

## **Creating Dataframes**

In [ ]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [ ]:
target = "exam_score"
drop_cols = [target, "id"]

X = train.drop(columns=drop_cols)
y = train[target]

## **Feature Engineering**

In [ ]:
class AddFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Formula 1 (from 1st place)
        X['engineered_score_v1'] = (
            5.760845 * X['study_hours'] +
            4.760845 * (X['sleep_quality'] == 'average').astype(int) +
            8.760845 * (X['sleep_quality'] == 'good').astype(int) +
            9.52169  * (X['study_method'] == 'coaching').astype(int) +
            0.325220702899505 * X['class_attendance'] +
            1.0      * (X['study_method'] == 'group study').astype(int) +
            4.760845 * (X['study_method'] == 'mixed').astype(int) +
            5.760845 * (X['facility_rating'] == 'high').astype(int) -
            1.0      * (X['facility_rating'] == 'low').astype(int) +
            2.0      * (X['facility_rating'] == 'medium').astype(int) +
            X['sleep_hours']
        )

        # Formula 2 (from 1st place)
        X['engineered_score_v2'] = (
            5.87666929874987 * X['study_hours'] +
            5.0      * (X['sleep_quality'] == 'average').astype(int) +
            9.674877 * (X['sleep_quality'] == 'good').astype(int) +
            8.222835 * (X['study_method'] == 'coaching').astype(int) +
            0.327161004763791 * X['class_attendance'] +
            3.0      * (X['study_method'] == 'mixed').astype(int) +
            3.659515 * (X['facility_rating'] == 'high').astype(int) -
            4.222835 * (X['facility_rating'] == 'low').astype(int) +
            X['sleep_hours'] +
            2.020001
        )

        return X

In [ ]:
X_fe = AddFeatures().transform(X)

num_cols = X_fe.select_dtypes(include=["int64", "float64"]).columns
ordinal_cols = ["sleep_quality", "facility_rating", "exam_difficulty"]
cat_cols = X_fe.select_dtypes(include=["object", "bool"]).columns
cat_cols = cat_cols.drop(ordinal_cols)

## **Preprocessing**

### Numerical transformer

In [ ]:
numerical_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
])

### Categorical transformer

In [ ]:
categorical_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

### Ordinal transformer

In [ ]:
ordinal_transformer = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(categories=[
        ["poor", "average", "good"],
        ["low", "medium", "high"],
        ["easy", "moderate", "hard"]
    ]))
])

### Preprocessor

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols),
    ("ordinal", ordinal_transformer, ordinal_cols)
    ], remainder="passthrough"
)

## **Classifier**

In [ ]:
classifier = XGBRegressor(
    n_estimators=1000,
    max_depth=7,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=0.5,
    reg_alpha=0.0,
    min_child_weight=3
)

## **Pipeline**

In [ ]:
model = Pipeline(steps=[
    ("feature engineering", AddFeatures()),
    ("preprocessing", preprocessor),
    ("classifier", classifier)
])

## **Grid Search Cross Validation**

In [ ]:
param_grid = {
    'classifier__learning_rate': [0.05, 0.5],
    'classifier__max_depth': [5]
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=3,
    n_jobs=-1,
    verbose=2
)

grid_search.fit(X, y)
print(grid_search.best_params_)
print(grid_search.best_score_)

Fitting 3 folds for each of 2 candidates, totalling 6 fits


KeyboardInterrupt: 

## **Cross Validation**

In [ ]:
scores = cross_val_score(
    model,
    X,
    y,
    cv=3,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)
print("scores:", scores)
print("mean score:", scores.mean())

scores: [-8.77235407 -8.74670267 -8.7805137 ]
mean score: -8.766523480789024


## **Training**

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)

Pipeline(steps=[('feature engineering', AddFeatures()),
                ('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('impute',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['age', 'study_hours', 'class_attendance', 'sleep_hours',
       'engineered_score_v1', 'engineered_score_v2'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('impute',...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.05,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=7, max_leaves=None, min_child_weight=3,
                              missing=nan, monotone_constraints=None,
                              multi_strategy=None, n_estimators=1000,
                              n_jobs=None, num_parallel_tree=None, ...))])

## **Permutation Importance**

In [ ]:
# Doesnt check custom features
result = permutation_importance(
    model,
    X_val,
    y_val,
    n_repeats=3,
    random_state=42,
    scoring="neg_root_mean_squared_error"
)

perm_df = pd.DataFrame({
    "feature": X_val.columns,
    "importance": result.importances_mean
}).sort_values("importance", ascending=False)

print(perm_df.head(15))

             feature  importance
3        study_hours   12.218751
4   class_attendance    2.967255
7      sleep_quality    1.419514
8       study_method    1.193549
9    facility_rating    0.953847
6        sleep_hours    0.603947
2             course    0.008499
0                age    0.007300
1             gender    0.001794
5    internet_access    0.000939
10   exam_difficulty    0.000546


In [ ]:
"""
A quick check to see whether or not a feature should be removed
is if the error is smaller than the standard deviation
"""
print(result.importances_std[X_val.columns.get_loc("exam_difficulty")])

0.00020591559892855414


## **Evaluation**

In [ ]:
y_preds = model.predict(X_val)
root_mean_squared_error(y_val, y_preds)

8.72941254342328

## **Submission**

In [ ]:
submission = pd.read_csv("data/sample_submission.csv")
submission["exam_score"] = model.predict(test)
submission.to_csv("submission.csv", index=False)

## **Reflections**

**Notes**
*   `study_hours` and `class_attendance` were the most important features for the model.
*   The model ended up achieving a public score of 8.708 which places me number #928 out of the 4317 teams.

**What i learned**
*   how to implement XGBoost for regression
*   how to use Grid Search to find optimal hyperparameter choices.
*   What genetic programming is and how it can be useful to create features.
*   Ordinal encoding and when it is useful
*   new scoring metric: neg_root_mean_squared_error








